# 05 - Treinamento do soft sensor

**Objetivo:** Comparar baseline, regressão linear, Ridge e Random Forest com validação temporal.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor
import joblib
base=carregar_csv(ROOT/'dados/03_processados/base_soft_sensor.csv'); target='temperatura_interna_C'; features=[c for c in base.columns if c not in ['timestamp',target,'evento_referencia','particao']]
tr=base.particao.eq('treino'); va=base.particao.eq('validacao'); te=base.particao.eq('teste')

## Treinamento e seleção

In [ ]:
models={'Regressao linear':Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('m',LinearRegression())]),'Ridge':Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('m',Ridge(alpha=2.0))]),'Random Forest':Pipeline([('imp',SimpleImputer(strategy='median')),('m',RandomForestRegressor(n_estimators=160,max_depth=14,min_samples_leaf=2,n_jobs=-1,random_state=42))])}
rows=[]
# Baseline climatológico simples
for name,m in models.items():
    m.fit(base.loc[tr,features],base.loc[tr,target]); p=m.predict(base.loc[va,features]); rows.append({'modelo':name,**metricas_regressao(base.loc[va,target],p)})
resval=pd.DataFrame(rows).sort_values('MAE'); display(resval); best_name=resval.iloc[0].modelo; best=models[best_name]; print('Selecionado:',best_name)

## Avaliação no teste e por cenário

In [ ]:
pred=best.predict(base.loc[te,features]); y=base.loc[te,target]; geral=metricas_regressao(y,pred); print(geral)
final=pd.DataFrame({'timestamp':base.loc[te,'timestamp'],'observado':y,'estimado':pred,'evento':base.loc[te,'evento_referencia']}); final['residuo']=final.estimado-final.observado
cen=final.groupby('evento').apply(lambda x:pd.Series(metricas_regressao(x.observado,x.estimado)),include_groups=False).reset_index(); display(cen)
pd.DataFrame([{'modelo':best_name,**geral}]).to_csv(ROOT/'resultados/metricas/metricas_soft_sensor.csv',index=False); cen.to_csv(ROOT/'resultados/metricas/metricas_soft_sensor_por_evento.csv',index=False)

## Gráficos e modelo

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,5)); axs[0].scatter(final.observado,final.estimado,s=8,alpha=.3); lo=min(final.observado.min(),final.estimado.min()); hi=max(final.observado.max(),final.estimado.max()); axs[0].plot([lo,hi],[lo,hi],'r--'); axs[0].set(xlabel='Observado °C',ylabel='Estimado °C')
axs[1].hist(final.residuo,bins=40); axs[1].set(xlabel='Resíduo °C',ylabel='Frequência'); fig.tight_layout(); fig.savefig(ROOT/'resultados/figuras/soft_sensor/avaliacao_soft_sensor.png',dpi=150); plt.show()
joblib.dump({'modelo':best,'features':features,'nome':best_name},ROOT/'resultados/modelos/soft_sensor.joblib'); final.to_csv(ROOT/'dados/03_processados/predicoes_soft_sensor_teste.csv',index=False); print('Modelo salvo.')

## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.